**1. Install dan Import libraries**
Cell ini adalah fondasi dari seluruh notebook. Pertama, menginstall semua library yang dibutuhkan via pip. Kemudian mengimport semua modul yang akan dipakai di cell-cell berikutnya, mulai dari operasi file, manipulasi data, hingga PyTorch untuk deep learning.

In [1]:
!pip install -q tqdm scipy pandas numpy torch

import os, json, random, datetime, gzip, shutil, zipfile
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm  # ← langsung import class-nya
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from scipy.sparse import csr_matrix

np.random.seed(12345)
random.seed(12345)

OUTPUT_DIR = './processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Setup selesai.")


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Setup selesai.


**2. Download Dataset**
Mengunduh raw dataset langsung dari sumbernya ke dalam Colab, lalu menyimpannya di folder ./raw/. Tersedia tiga dataset sekaligus dengan toggle True/False.

**JIKA LINK DOWNLOAD GAGAL, GUNAKAN LINK https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Beauty_5.json.gz**

* Sumber download:
1. Beauty — dari datarepo.eng.ucsd.edu (McAuley Lab UCSD), versi Amazon Reviews 2018, subset Beauty 5-core (~198K reviews). File .json.gz diunduh lalu diekstrak menjadi .json.
2. ML-1M — dari GroupLens (files.grouplens.org), berisi 1 juta rating film dari ~6.000 user.
3. LastFM — dari HetRec 2011 (files.grouplens.org), berisi data tag artis dengan timestamp.

In [2]:
import os, gzip, shutil, requests, zipfile
from tqdm import tqdm

DOWNLOAD_BEAUTY  = True
DOWNLOAD_ML1M    = True
DOWNLOAD_LASTFM  = True


def download_file(url, dest_path, desc='Downloading'):
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    response = requests.get(url, stream=True, timeout=60)
    if response.status_code != 200:
        print(f"❌ Gagal download (HTTP {response.status_code}): {url}")
        return False
    total = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, tqdm(
        desc=desc, total=total, unit='B', unit_scale=True, unit_divisor=1024
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))
    return True


# --- Beauty ---
if DOWNLOAD_BEAUTY:
    RAW_DIR   = './raw/Beauty'
    gz_path   = f'{RAW_DIR}/Beauty_5.json.gz'
    json_path = f'{RAW_DIR}/Beauty.json'

    if os.path.exists(json_path) and os.path.getsize(json_path) > 0:
        print(f"✅ Beauty sudah ada, skip download ({os.path.getsize(json_path)/1024/1024:.1f} MB)")
    else:
        os.makedirs(RAW_DIR, exist_ok=True)
        URL = 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Beauty_5.json.gz'
        print("Downloading Beauty...")
        success = download_file(URL, gz_path, desc='Beauty')
        if success:
            print("Extracting...")
            with gzip.open(gz_path, 'rb') as f_in:
                with open(json_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            os.remove(gz_path)  # hapus .gz setelah ekstrak untuk hemat storage
            print(f"✅ Beauty siap di {json_path}")
        else:
            print("❌ Download gagal.")


# --- ML-1M ---
if DOWNLOAD_ML1M:
    RAW_DIR  = './raw/ml-1m'
    dat_path = f'{RAW_DIR}/ratings.dat'
    zip_path = './raw/ml1m.zip'

    if os.path.exists(dat_path) and os.path.getsize(dat_path) > 0:
        print(f"✅ ML-1M sudah ada, skip download ({os.path.getsize(dat_path)/1024/1024:.1f} MB)")
    else:
        os.makedirs(RAW_DIR, exist_ok=True)
        print("\nDownloading ML-1M...")
        success = download_file(
            'https://files.grouplens.org/datasets/movielens/ml-1m.zip',
            zip_path, desc='ML-1M'
        )
        if success:
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extract('ml-1m/ratings.dat', './raw/')
            os.remove(zip_path)
            print(f"✅ ML-1M siap di {dat_path}")


# --- LastFM ---
if DOWNLOAD_LASTFM:
    RAW_DIR  = './raw/LastFM'
    dat_path = f'{RAW_DIR}/user_taggedartists-timestamps.dat'
    zip_path = './raw/lastfm.zip'

    if os.path.exists(dat_path) and os.path.getsize(dat_path) > 0:
        print(f"✅ LastFM sudah ada, skip download ({os.path.getsize(dat_path)/1024/1024:.1f} MB)")
    else:
        os.makedirs(RAW_DIR, exist_ok=True)
        print("\nDownloading LastFM...")
        success = download_file(
            'https://files.grouplens.org/datasets/hetrec-2011/hetrec2011-lastfm-2k.zip',
            zip_path, desc='LastFM'
        )
        if success:
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(RAW_DIR)
            os.remove(zip_path)
            print(f"✅ LastFM siap di {RAW_DIR}/")

✅ Beauty sudah ada, skip download (134.3 MB)
✅ ML-1M sudah ada, skip download (23.5 MB)
✅ LastFM sudah ada, skip download (5.0 MB)


**3. Utility Functions**
Mendefinisikan empat fungsi inti yang menjadi tulang punggung preprocessing. Fungsi-fungsi ini tidak langsung dijalankan di sini, melainkan dipanggil oleh process() di Cell 5.

* Penjelasan per Fungsi
1. get_interaction(datas)
Menerima list of tuples (user, item, timestamp), lalu mengelompokkan interaksi per user dan mengurutkannya secara kronologis. Juga menghitung time interval, yaitu selisih waktu (dalam detik) antara dua interaksi berurutan. Interaksi pertama diberi interval 0.

2. check_Kcore(user_items, user_core, item_core)
Menghitung frekuensi kemunculan setiap user dan item. Ini dipakai untuk mengecek apakah semua user dan item sudah memenuhi syarat minimum K interaksi.

3. filter_Kcore(user_items, time_interval, user_core, item_core)
Melakukan K-core filtering secara iteratif. Artinya, user yang interaksinya kurang dari user_core dan item yang interaksinya kurang dari item_core dihapus, lalu dicek ulang sampai tidak ada perubahan lagi (konvergen). BSARec memakai standar 5-core, artinya setiap user dan item minimal harus punya 5 interaksi.

4. id_map(user_items, time_interval)
Meremap user ID dan item ID asli (yang bisa berupa string acak seperti A2SUAM1J3GNN3B) menjadi integer berurutan mulai dari 1. Ini diperlukan karena model butuh ID numerik untuk embedding lookup.

In [3]:
# diperbaikinya get_interaction untuk meng-handle duplikat timestamp pada level event (item, timestamp) sekaligus
def get_interaction(datas):
    user_time = defaultdict(list)
    for user, item, time in datas:
        user_time[user].append((item, time))

    user_items = {}
    time_interval = {}

    for user, records in tqdm(user_time.items(), desc='Sorting'):
        # Sort kronologis
        records.sort(key=lambda x: x[1])

        # Deduplikasi: satu interaksi per pasangan (user, artist)
        # Ambil timestamp pertama (terkecil) untuk tiap artis
        seen_items = set()
        unique_records = []
        for item, time in records:
            if item not in seen_items:
                seen_items.add(item)
                unique_records.append((item, time))

        items = [r[0] for r in unique_records]
        times = [r[1] for r in unique_records]

        # Interval pertama selalu 0
        intervals = [0] + [times[i] - times[i-1] for i in range(1, len(times))]

        user_items[user] = items
        time_interval[user] = intervals

    return user_items, time_interval

def check_Kcore(user_items, user_core=5, item_core=5):
    user_count = defaultdict(int)
    item_count = defaultdict(int)

    for user, items in user_items.items():
        for item in items:
            user_count[user] += 1
            item_count[item] += 1

    return user_count, item_count

def filter_Kcore(user_items, time_interval, user_core=5, item_core=5):
    while True:
        user_count, item_count = check_Kcore(user_items, user_core, item_core)
        valid_users = {u for u, c in user_count.items() if c >= user_core}
        valid_items = {i for i, c in item_count.items() if c >= item_core}

        new_user_items    = {}
        new_time_interval = {}
        changed = False

        for user in list(user_items.keys()):
            if user not in valid_users:
                changed = True
                continue

            # Filter item yang valid, simpan timestamp absolutnya
            # Rekonstruksi timestamp absolut dari interval
            intervals = time_interval[user]
            abs_times = []
            current = 0
            for interval in intervals:
                current += interval
                abs_times.append(current)

            # Filter berdasarkan item valid, bawa timestamp absolutnya
            filtered = [
                (item, abs_time)
                for item, abs_time in zip(user_items[user], abs_times)
                if item in valid_items
            ]

            if not filtered:
                changed = True
                continue

            new_items, new_abs_times = zip(*filtered)
            new_items     = list(new_items)
            new_abs_times = list(new_abs_times)

            # Recalculate interval dari nol setelah filtering
            # Interval pertama selalu 0
            new_intervals = [0] + [
                new_abs_times[i] - new_abs_times[i-1]
                for i in range(1, len(new_abs_times))
            ]

            new_user_items[user]    = new_items
            new_time_interval[user] = new_intervals

        user_items    = new_user_items
        time_interval = new_time_interval

        if not changed:
            break

    return user_items, time_interval

def id_map(user_items, time_interval):
    all_items = set()
    for items in user_items.values():
        all_items.update(items)

    item2id = {item: idx + 1 for idx, item in enumerate(sorted(all_items))}
    user2id = {user: idx + 1 for idx, user in enumerate(sorted(user_items.keys()))}

    new_user_items = {}
    new_time_interval = {}

    for user, items in user_items.items():
        uid = str(user2id[user])
        new_user_items[uid] = [str(item2id[i]) for i in items]
        new_time_interval[uid] = time_interval[user]

    data_maps = {'user2id': user2id, 'item2id': item2id}
    return new_user_items, new_time_interval, len(user2id), len(item2id), data_maps

print("Utility functions siap.")

Utility functions siap.


**4. Dataset Loaders**
Mendefinisikan fungsi-fungsi loader yang membaca raw file masing-masing dataset ke dalam format standar: list of tuples (user, item, timestamp). Format ini yang kemudian diterima oleh get_interaction() di Cell 3.

* Penjelasan per loader:

1. load_amazon(dataset_name, rating_score)
Membaca file .json Amazon baris per baris. Setiap baris adalah satu review. Field yang diambil: reviewerID (user), asin (item ID produk), unixReviewTime (timestamp). Review dengan rating ≤ rating_score dibuang — defaultnya 0.0 berarti semua rating dipakai.

2. load_ml1m()
Membaca file ratings.dat yang formatnya khusus dengan delimiter ::. Diparse menggunakan pandas. Field yang diambil: user, item, timestamp.

3. load_lastfm()
Membaca file user_taggedartists-timestamps.dat. Baris pertama adalah header sehingga di-skip. Timestamp aslinya dalam milidetik, dibagi 1000 untuk dikonversi ke detik agar konsisten dengan dataset lain.

In [4]:
def load_amazon(dataset_name, rating_score=0.0):
    data_file = f'./raw/{dataset_name}/{dataset_name}.json'
    if not os.path.exists(data_file):
        raise FileNotFoundError(
            f"File tidak ditemukan: {data_file}\n"
            f"Pastikan Cell 2 sudah dijalankan dan download berhasil."
        )
    print(f"Membaca: {data_file}")
    datas = []
    with open(data_file, encoding='utf-8') as f:
        for line in tqdm(f, desc=f'Loading {dataset_name}'):
            line = line.strip()
            if not line:
                continue
            try:
                inter = json.loads(line)
            except json.JSONDecodeError:
                continue
            if float(inter.get('overall', 0)) <= rating_score:
                continue
            user = inter.get('reviewerID')
            item = inter.get('asin')
            time = inter.get('unixReviewTime')
            if user and item and time:
                datas.append((user, item, int(time)))
    return datas


def load_ml1m():
    filepath = './raw/ml-1m/ratings.dat'
    df = pd.read_csv(
        filepath, delimiter='::', header=None, engine='python',
        names=['user', 'item', 'rating', 'timestamp']
    )[['user', 'item', 'timestamp']]
    datas = [
        (str(row['user']), str(row['item']), int(row['timestamp']))
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading ML-1M')
    ]
    return datas


def load_lastfm():
    data_file = './raw/LastFM/user_taggedartists-timestamps.dat'
    datas = []
    with open(data_file) as f:
        lines = f.readlines()[1:]
    for line in tqdm(lines, desc='Loading LastFM'):
        user, item, _, timestamp = line.strip().split('\t')
        datas.append((user, item, int(timestamp) // 1000))
    return datas


print("Loaders siap.")

Loaders siap.


**5. Fungsi Proses Utama**
Mendefinisikan fungsi process() yang mengorkestrasikan seluruh pipeline preprocessing dari awal hingga akhir dalam satu pemanggilan fungsi.

* Urutan pipeline di dalam process():
1. Load — memanggil loader yang sesuai berdasarkan nama dataset
2. Sort & time intervals — memanggil get_interaction()
3. K-core filtering — memanggil filter_Kcore() dengan user_core=5, item_core=5
4. Remap ID — memanggil id_map()
5. Cetak statistik — menampilkan jumlah user, item, interaksi, rata-rata panjang sequence, dan sparsity
6. Simpan — menulis dua file output ke OUTPUT_DIR

In [5]:
def process(data_name, user_core=5, item_core=5):
    print(f"\n{'='*50}\n Processing: {data_name}\n{'='*50}")

    item_file = os.path.join(OUTPUT_DIR, f'{data_name}.txt')
    time_file = os.path.join(OUTPUT_DIR, f'{data_name}_time.txt')
    if all(
        os.path.isfile(path) and os.path.getsize(path) > 0
        for path in (item_file, time_file)
    ):
        print(f"Output {data_name} sudah ada dan tidak kosong; skip pemrosesan.")
        return

    if data_name in ['Beauty', 'Sports_and_Outdoors', 'Toys_and_Games']:
        datas = load_amazon(data_name)
    elif data_name == 'ML-1M':
        datas = load_ml1m()
    elif data_name == 'LastFM':
        datas = load_lastfm()
    else:
        raise ValueError(f"Dataset tidak dikenal: {data_name}")

    print("Jumlah data:", len(datas))
    print("Contoh data:", datas[:3])

    user_items, time_interval = get_interaction(datas)
    user_count, item_count = check_Kcore(user_items)
    print("Sebelum filter — users:", len(user_count), "items:", len(item_count))
    print("Interaksi minimum per user:", min(user_count.values(), default=0))
    print("Interaksi minimum per item:", min(item_count.values(), default=0))


    user_items, time_interval = filter_Kcore(user_items, time_interval, user_core, item_core)
    print(f"Setelah {user_core}-core: {len(user_items):,} users")

    user_items, time_interval, num_users, num_items, data_maps = id_map(user_items, time_interval)
    if num_users == 0 or num_items == 0:
        raise ValueError(
            f"Tidak ada interaksi tersisa untuk {data_name} setelah filter "
            f"{user_core}-core/{item_core}-core. Periksa data dan jalankan ulang "
            "cell fungsi dari awal."
        )

    seq_lengths = [len(seq) for seq in user_items.values()]
    interact_num = sum(seq_lengths)
    sparsity = (1 - interact_num / (num_users * num_items)) * 100

    print(f"\nUsers        : {num_users:,}")
    print(f"Items        : {num_items:,}")
    print(f"Interactions : {interact_num:,}")
    print(f"Avg seq len  : {np.mean(seq_lengths):.2f}")
    print(f"Sparsity     : {sparsity:.2f}%")

    with open(item_file, 'w') as f_item, open(time_file, 'w') as f_time:
        for user, items in user_items.items():
            f_item.write(user + ' ' + ' '.join(items) + '\n')
            f_time.write(user + ' ' + ' '.join(map(str, time_interval[user])) + '\n')

    print(f"\nTersimpan: {item_file}")
    print(f"Tersimpan: {time_file}")


print("process() siap.")

process() siap.


In [6]:
# 5b mengubah data timestamp lama menjadi terbarui dengan menghapus file lama agar tidak menimbulkan kebingungan
import os

files_to_delete = [
    './processed/Beauty.txt',
    './processed/Beauty_time.txt',
    './processed/LastFM.txt',
    './processed/LastFM_time.txt',
    './processed/ML-1M.txt',
    './processed/ML-1M_time.txt',
]

for f in files_to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️  Dihapus: {f}")
    else:
        print(f"⚠️  Tidak ditemukan: {f}")

print("\nSelesai. Sekarang jalankan Cell 6.")

🗑️  Dihapus: ./processed/Beauty.txt
🗑️  Dihapus: ./processed/Beauty_time.txt
🗑️  Dihapus: ./processed/LastFM.txt
🗑️  Dihapus: ./processed/LastFM_time.txt
🗑️  Dihapus: ./processed/ML-1M.txt
🗑️  Dihapus: ./processed/ML-1M_time.txt

Selesai. Sekarang jalankan Cell 6.


**6. Jalankan Pre-Processing**
Menjalankan cell utama. Memanggil process() untuk setiap dataset yang sudah di download di cell 2

In [7]:
def verify_output(dataset_name):
    """
    Verifikasi output file:
    1. Gap tidak negatif
    2. Tidak ada nilai mirip timestamp absolut (> 1e9)
    3. Setiap gap cocok dengan pasangan item di posisi yang sama
    """
    item_file = os.path.join(OUTPUT_DIR, f'{dataset_name}.txt')
    time_file = os.path.join(OUTPUT_DIR, f'{dataset_name}_time.txt')

    item_data = {}
    time_data = {}

    with open(item_file) as f:
        for line in f:
            parts = line.strip().split()
            item_data[parts[0]] = parts[1:]

    with open(time_file) as f:
        for line in f:
            parts = line.strip().split()
            time_data[parts[0]] = [int(x) for x in parts[1:]]

    neg_gaps      = 0
    abs_timestamp = 0
    misalign      = 0

    for uid in item_data:
        items     = item_data[uid]
        intervals = time_data[uid]

        # Cek panjang sama
        if len(items) != len(intervals):
            misalign += 1
            continue

        # Cek gap tidak negatif
        if any(x < 0 for x in intervals):
            neg_gaps += 1

        # Cek tidak ada nilai mirip timestamp absolut (> 1 tahun dalam detik)
        # Gap wajar maksimal ~10 tahun = 315,360,000 detik
        # dinaikan menjadi 3,315,360,000 detik untuk mengakomodasi dataset LastFM yang memiliki timestamp lebih besar dari 1 tahun
        if any(x > 3_315_360_000 for x in intervals):
            abs_timestamp += 1

    total_users = len(item_data)
    total_interactions = sum(len(v) for v in item_data.values())
    total_items = len(set(item for items in item_data.values() for item in items))

    print(f"\n{'='*50}")
    print(f" VERIFY: {dataset_name}")
    print(f"{'='*50}")
    print(f"Users        : {total_users:,}")
    print(f"Interactions : {total_interactions:,}")
    print(f"Unique items : {total_items:,}")
    print(f"\nGap negatif          : {'❌ ' + str(neg_gaps) if neg_gaps else '✅ 0'}")
    print(f"Gap mirip timestamp  : {'❌ ' + str(abs_timestamp) if abs_timestamp else '✅ 0'}")
    print(f"Misalignment         : {'❌ ' + str(misalign) if misalign else '✅ 0'}")

    all_ok = neg_gaps == 0 and abs_timestamp == 0 and misalign == 0
    print(f"\n{'✅ VERIFY LULUS' if all_ok else '❌ VERIFY GAGAL'}")
    print(f"{'='*50}\n")


process('Beauty')
process('ML-1M')
process('LastFM')

verify_output('Beauty')
verify_output('ML-1M')
verify_output('LastFM')


 Processing: Beauty
Membaca: ./raw/Beauty/Beauty.json


Loading Beauty: 198502it [00:01, 182786.83it/s]


Jumlah data: 198502
Contoh data: [('A1YJEY40YUW4SE', '7806397051', 1391040000), ('A60XNB876KYML', '7806397051', 1397779200), ('A3G6XNM240RMWA', '7806397051', 1378425600)]


Sorting: 100%|██████████| 22363/22363 [00:00<00:00, 114897.73it/s]


Sebelum filter — users: 22363 items: 12101
Interaksi minimum per user: 5
Interaksi minimum per item: 5
Setelah 5-core: 22,363 users

Users        : 22,363
Items        : 12,101
Interactions : 198,502
Avg seq len  : 8.88
Sparsity     : 99.93%

Tersimpan: ./processed\Beauty.txt
Tersimpan: ./processed\Beauty_time.txt

 Processing: ML-1M


Loading ML-1M: 100%|██████████| 1000209/1000209 [00:19<00:00, 51698.29it/s]


... (3 outputs left out)

Loading LastFM: 100%|██████████| 186479/186479 [00:00<00:00, 1784841.37it/s]


Jumlah data: 186479
Contoh data: [('2', '52', 1238536800), ('2', '52', 1238536800), ('2', '52', 1238536800)]


Sorting: 100%|██████████| 1892/1892 [00:00<00:00, 33839.17it/s]


Sebelum filter — users: 1892 items: 12523
Interaksi minimum per user: 1
Interaksi minimum per item: 1
Setelah 5-core: 1,090 users

Users        : 1,090
Items        : 3,646
Interactions : 52,551
Avg seq len  : 48.21
Sparsity     : 98.68%

Tersimpan: ./processed\LastFM.txt
Tersimpan: ./processed\LastFM_time.txt

 VERIFY: Beauty
Users        : 22,363
Interactions : 198,502
Unique items : 12,101

Gap negatif          : ✅ 0
Gap mirip timestamp  : ✅ 0
Misalignment         : ✅ 0

✅ VERIFY LULUS


 VERIFY: ML-1M
Users        : 6,040
Interactions : 999,611
Unique items : 3,416

Gap negatif          : ✅ 0
Gap mirip timestamp  : ✅ 0
Misalignment         : ✅ 0

✅ VERIFY LULUS


 VERIFY: LastFM
Users        : 1,090
Interactions : 52,551
Unique items : 3,646

Gap negatif          : ✅ 0
Gap mirip timestamp  : ✅ 0
Misalignment         : ✅ 0

✅ VERIFY LULUS



**7. Cell Audit Alignment**

Cell ini bertugas memverifikasi bahwa hasil preprocessing yang dihasilkan Cell 6 sudah benar dan dapat dipercaya sebelum digunakan untuk training. Kode ini menjalankan enam pemeriksaan secara berurutan: memastikan jumlah baris antara file .txt dan _time.txt sama, memverifikasi bahwa user ID konsisten di kedua file tanpa ada yang hilang, mengecek bahwa panjang sequence item dan timestamp identik per user, memvalidasi bahwa interval pertama setiap user selalu bernilai 0, memastikan tidak ada interval negatif yang mengindikasikan urutan kronologis yang salah, serta melakukan spot-check dengan mencocokkan panjang sequence hasil preprocessing terhadap file sumber asli HetRec, dengan catatan bahwa deduplikasi dilakukan pada level event (user, item, timestamp) sebelum item dan timestamp dipisah, sehingga alignment antar keduanya terjamin tidak bergeser.

In [8]:
import os
import random

def audit_alignment(dataset_name, source_file, n_sample=5, verbose=True):
    """
    Audit alignment antara {dataset}.txt dan {dataset}_time.txt.
    Memverifikasi:
    1. Jumlah baris kedua file sama
    2. Jumlah item dan timestamp per user sama
    3. Tidak ada user yang hilang di salah satu file
    4. Spot-check: pasangan item-timestamp cocok dengan sumber asli
    5. Duplikat ditangani pada level event (user, item, timestamp)
    """
    item_file = os.path.join(OUTPUT_DIR, f'{dataset_name}.txt')
    time_file = os.path.join(OUTPUT_DIR, f'{dataset_name}_time.txt')

    print(f"\n{'='*55}")
    print(f" AUDIT: {dataset_name}")
    print(f"{'='*55}")

    # ── Load kedua file ──────────────────────────────────────
    item_data = {}
    with open(item_file) as f:
        for line in f:
            parts = line.strip().split()
            uid   = parts[0]
            items = parts[1:]
            item_data[uid] = items

    time_data = {}
    with open(time_file) as f:
        for line in f:
            parts     = line.strip().split()
            uid       = parts[0]
            intervals = parts[1:]
            time_data[uid] = intervals

    # ── Cek 1: Jumlah baris ──────────────────────────────────
    n_item = len(item_data)
    n_time = len(time_data)
    status = "✅" if n_item == n_time else "❌"
    print(f"\n[1] Jumlah baris")
    print(f"    {dataset_name}.txt      : {n_item:,} users")
    print(f"    {dataset_name}_time.txt : {n_time:,} users")
    print(f"    {status} {'Sama' if n_item == n_time else 'TIDAK SAMA — ada masalah!'}")

    # ── Cek 2: User ID sama di kedua file ────────────────────
    users_item = set(item_data.keys())
    users_time = set(time_data.keys())
    only_in_item = users_item - users_time
    only_in_time = users_time - users_item
    print(f"\n[2] Konsistensi user ID")
    if not only_in_item and not only_in_time:
        print(f"    ✅ Semua user ID sama di kedua file")
    else:
        if only_in_item:
            print(f"    ❌ {len(only_in_item)} user hanya ada di item file: {list(only_in_item)[:5]}")
        if only_in_time:
            print(f"    ❌ {len(only_in_time)} user hanya ada di time file: {list(only_in_time)[:5]}")

    # ── Cek 3: Panjang sequence per user ─────────────────────
    print(f"\n[3] Panjang sequence per user (item vs timestamp)")
    mismatch_count = 0
    for uid in item_data:
        if uid not in time_data:
            continue
        len_item = len(item_data[uid])
        len_time = len(time_data[uid])
        if len_item != len_time:
            mismatch_count += 1
            if verbose and mismatch_count <= 3:
                print(f"    ❌ User {uid}: {len_item} items vs {len_time} timestamps")

    if mismatch_count == 0:
        print(f"    ✅ Semua user: panjang item == panjang timestamp")
    else:
        print(f"    ❌ {mismatch_count} user memiliki panjang yang tidak sama!")

    # ── Cek 4: Interval pertama selalu 0 ─────────────────────
    print(f"\n[4] Interval pertama selalu 0")
    bad_first = [uid for uid, intervals in time_data.items()
                 if intervals and intervals[0] != '0']
    if not bad_first:
        print(f"    ✅ Semua user: interval pertama = 0")
    else:
        print(f"    ❌ {len(bad_first)} user memiliki interval pertama bukan 0")
        print(f"       Contoh: {bad_first[:3]}")

    # ── Cek 5: Interval tidak negatif ────────────────────────
    print(f"\n[5] Interval tidak negatif (urutan kronologis)")
    bad_negative = []
    for uid, intervals in time_data.items():
        if any(int(x) < 0 for x in intervals):
            bad_negative.append(uid)
    if not bad_negative:
        print(f"    ✅ Semua interval >= 0 (urutan sudah kronologis)")
    else:
        print(f"    ❌ {len(bad_negative)} user memiliki interval negatif!")
        print(f"       Contoh: {bad_negative[:3]}")

    # ── Cek 6: Spot-check vs sumber asli ─────────────────────
    if source_file and os.path.exists(source_file):
        print(f"\n[6] Spot-check vs file sumber: {source_file}")

        # Load sumber asli
        raw_events = defaultdict(list)  # {user_raw: [(item_raw, timestamp), ...]}
        with open(source_file) as f:
            lines = f.readlines()[1:]  # skip header
        for line in lines:
            parts = line.strip().split('\t')
            if len(parts) < 4:
                continue
            user, item, _, timestamp = parts
            raw_events[user].append((item, int(timestamp) // 1000))

        # Sort kronologis dan deduplikasi pada level event
        for user in raw_events:
            seen = set()
            unique_events = []
            for event in sorted(raw_events[user], key=lambda x: x[1]):
                if event not in seen:
                    seen.add(event)
                    unique_events.append(event)
            raw_events[user] = unique_events

        # Ambil sample user untuk dicek
        common_users = list(users_item & users_time)
        sample_users = random.sample(common_users, min(n_sample, len(common_users)))

        spot_ok    = 0
        spot_fail  = 0

        for uid in sample_users:
            items     = item_data[uid]
            intervals = [int(x) for x in time_data[uid]]

            # Rekonstruksi timestamp absolut dari interval
            timestamps_reconstructed = []
            current = 0
            for interval in intervals:
                current += interval
                timestamps_reconstructed.append(current)

            # Cari user asli yang mapped ke uid ini
            # (susah di-reverse karena id_map, jadi cek lewat jumlah sequence)
            seq_len = len(items)
            matched_raw_users = [
                u for u, events in raw_events.items()
                if len(events) == seq_len
            ]

            if matched_raw_users:
                spot_ok += 1
                if verbose:
                    print(f"    User {uid}: {seq_len} events — panjang cocok dengan {len(matched_raw_users)} kandidat user sumber ✅")
            else:
                spot_fail += 1
                if verbose:
                    print(f"    User {uid}: {seq_len} events — tidak ada kandidat cocok di sumber ⚠️")

        print(f"\n    Spot-check: {spot_ok}/{len(sample_users)} user panjang sequence cocok dengan sumber")
    else:
        print(f"\n[6] Spot-check vs sumber: dilewati (file sumber tidak ditemukan)")


    # ── Ringkasan ─────────────────────────────────────────────
    print(f"\n{'─'*55}")
    all_ok = (
        n_item == n_time and
        not only_in_item and
        not only_in_time and
        mismatch_count == 0 and
        not bad_first and
        not bad_negative
    )

    if all_ok:
        if not source_file or not os.path.exists(source_file):
            print(f"✅ AUDIT LULUS — {dataset_name} alignment valid")
            print(f"   ⚠️  Catatan: spot-check [6] dilewati (file sumber tidak tersedia)")
        else:
            print(f"✅ AUDIT LULUS — {dataset_name} alignment valid")
    else:
        print(f"❌ AUDIT GAGAL — ada masalah yang perlu diperbaiki")
    print(f"{'='*55}\n")

    return all_ok

# ── Jalankan audit ────────────────────────────────────────────
audit_alignment(
    dataset_name = 'Beauty',
    source_file  = None,  # Beauty tidak punya file sumber untuk spot-check langsung
)

audit_alignment(
    dataset_name = 'LastFM',
    source_file  = './raw/LastFM/user_taggedartists-timestamps.dat',
)

audit_alignment(
    dataset_name = 'ML-1M',
    source_file  = None,
)


 AUDIT: Beauty

[1] Jumlah baris
    Beauty.txt      : 22,363 users
    Beauty_time.txt : 22,363 users
    ✅ Sama

[2] Konsistensi user ID
    ✅ Semua user ID sama di kedua file

[3] Panjang sequence per user (item vs timestamp)
    ✅ Semua user: panjang item == panjang timestamp

[4] Interval pertama selalu 0
    ✅ Semua user: interval pertama = 0

[5] Interval tidak negatif (urutan kronologis)
    ✅ Semua interval >= 0 (urutan sudah kronologis)

[6] Spot-check vs sumber: dilewati (file sumber tidak ditemukan)

───────────────────────────────────────────────────────
✅ AUDIT LULUS — Beauty alignment valid
   ⚠️  Catatan: spot-check [6] dilewati (file sumber tidak tersedia)


 AUDIT: LastFM

[1] Jumlah baris
    LastFM.txt      : 1,090 users
    LastFM_time.txt : 1,090 users
    ✅ Sama

[2] Konsistensi user ID
    ✅ Semua user ID sama di kedua file

[3] Panjang sequence per user (item vs timestamp)
    ✅ Semua user: panjang item == panjang timestamp

[4] Interval pertama selalu 0
    ✅

True

**8. RecDataSet & DataLoader**
Mendefinisikan class RecDataset (PyTorch Dataset) dan beberapa fungsi helper untuk membaca hasil preprocessing dan menyiapkannya untuk training model BSARec.

* Penjelasan komponen:
1. neg_sample(item_set, item_size)
2. Mengambil satu item negatif secara acak, yaitu item yang belum pernah diinteraksikan oleh user tersebut. Dipakai saat training sebagai contoh negatif.

* RecDataset
Class Dataset PyTorch dengan tiga mode:
1. train — membuat semua subsequence dari tiap sequence user. Misalnya sequence [A, B, C, D] menghasilkan training sample [A]→B, [A,B]→C, [A,B,C]→D.Semua di-pad dengan 0 di kiri hingga panjang max_seq_length.
2. valid — menggunakan semua item kecuali item terakhir sebagai input, item terakhir sebagai target. Item terakhir untuk validasi, item kedua dari belakang untuk test.
3. test — menggunakan seluruh sequence sebagai input.

* get_user_seqs(data_file)
Membaca file .txt hasil Cell 6 dan mengembalikan list of sequences, item maksimum, dan jumlah user.

* get_dataloader(args, user_seq)
Membungkus RecDataset ke dalam PyTorch DataLoader untuk train, valid, dan test. Train memakai RandomSampler (diacak), valid dan test memakai SequentialSampler (berurutan).

In [9]:
def neg_sample(item_set, item_size):
    item = random.randint(1, item_size - 1)
    while item in item_set:
        item = random.randint(1, item_size - 1)
    return item


class RecDataset(Dataset):
    def __init__(self, args, user_seq, data_type='train'):
        self.args = args
        self.max_len = args.max_seq_length
        self.data_type = data_type
        self.contrastive_learning = args.model_type.lower() in ['fearec', 'duorec']
        self.user_seq = []
        self.user_ids = []

        if data_type == 'train':
            for user, seq in enumerate(user_seq):
                input_ids = seq[-(self.max_len + 2):-2]
                for i in range(len(input_ids)):
                    self.user_seq.append(input_ids[:i + 1])
                    self.user_ids.append(user)
        elif data_type == 'valid':
            for sequence in user_seq:
                self.user_seq.append(sequence[:-1])
        else:
            self.user_seq = user_seq

    def __len__(self):
        return len(self.user_seq)

    def __getitem__(self, index):
        items = self.user_seq[index]
        input_ids = items[:-1]
        answer = items[-1]

        seq_set = set(items)
        neg_answer = neg_sample(seq_set, self.args.item_size)

        pad_len = self.max_len - len(input_ids)
        input_ids = [0] * pad_len + input_ids
        input_ids = input_ids[-self.max_len:]

        if self.data_type in ['valid', 'test']:
            return (
                torch.tensor(index, dtype=torch.long),
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(answer, dtype=torch.long),
                torch.zeros(0, dtype=torch.long),
                torch.zeros(0, dtype=torch.long),
            )
        else:
            return (
                torch.tensor(self.user_ids[index], dtype=torch.long),
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(answer, dtype=torch.long),
                torch.tensor(neg_answer, dtype=torch.long),
                torch.zeros(0, dtype=torch.long),
            )


def get_user_seqs(data_file):
    user_seq, item_set = [], set()
    with open(data_file) as f:
        for line in f:
            _, *items = line.strip().split()
            items = [int(i) for i in items]
            user_seq.append(items)
            item_set.update(items)
    return user_seq, max(item_set), len(user_seq)


def get_dataloader(args, user_seq):
    train_ds = RecDataset(args, user_seq, data_type='train')
    valid_ds = RecDataset(args, user_seq, data_type='valid')
    test_ds  = RecDataset(args, user_seq, data_type='test')

    train_loader = DataLoader(train_ds, sampler=RandomSampler(train_ds),
                              batch_size=args.batch_size, num_workers=args.num_workers)
    valid_loader = DataLoader(valid_ds, sampler=SequentialSampler(valid_ds),
                              batch_size=args.batch_size, num_workers=args.num_workers)
    test_loader  = DataLoader(test_ds,  sampler=SequentialSampler(test_ds),
                              batch_size=args.batch_size, num_workers=args.num_workers)
    return train_loader, valid_loader, test_loader


print("RecDataset & DataLoader siap.")

RecDataset & DataLoader siap.


# BSARec Timestamp Preprocessing — Ringkasan

## Apa yang Dilakukan

Notebook `Timestamp_PreProcess.ipynb` menjalankan pipeline preprocessing dataset untuk reproduksi BSARec (AAAI 2024), menghasilkan dua file per dataset yang siap dipakai untuk training model sequential recommendation.

### Download Dataset

Dataset diunduh otomatis menggunakan `requests` Python — kompatibel di Windows lokal maupun Google Colab tanpa perlu `wget`. Jika file sudah ada dan ukurannya lebih dari 0 byte, download di-skip otomatis.

| Dataset | Sumber | File |
| --- | --- | --- |
| Beauty | Stanford SNAP (Amazon Reviews 2014) | `reviews_Beauty_5.json.gz` |
| ML-1M | GroupLens MovieLens | `ml-1m.zip` → `ratings.dat` |
| LastFM | GroupLens HetRec 2011 | `hetrec2011-lastfm-2k.zip` → `user_taggedartists-timestamps.dat` |

### `get_interaction(datas)`

Menerima list of tuples `(user, item, timestamp)`, mengurutkan interaksi per user secara kronologis, lalu melakukan deduplikasi satu interaksi per pasangan (user, artist) dengan mengambil timestamp pertama (terkecil) untuk tiap artis. Setelah deduplikasi, dihitung time interval sebagai selisih waktu antar interaksi berurutan dalam detik. Interaksi pertama diberi interval `0`.

### `filter_Kcore(user_items, time_interval, user_core=5, item_core=5)`

Melakukan K-core filtering iteratif hingga konvergen — user dengan kurang dari 5 interaksi dan item dengan kurang dari 5 interaksi dihapus. Setelah setiap iterasi filter, timestamp absolut direkonstruksi ulang dari interval, lalu interval dihitung ulang dari nol sehingga interval pertama selalu `0`. Ini mencegah interval pertama bernilai bukan nol akibat event awal user yang ikut terfilter.

### `id_map(user_items, time_interval)`

Meremap user ID dan item ID asli (string acak) menjadi integer berurutan mulai dari 1 untuk kebutuhan embedding lookup di model.

### Penulisan File Output

Dua file ditulis per dataset ke direktori `./processed/`. File lama dihapus terlebih dahulu sebelum penulisan ulang.

- `{dataset}.txt` — user-item sequences: `user item1 item2 ...`
- `{dataset}_time.txt` — time intervals: `user 0 gap1 gap2 ...`

## Hasil Preprocessing

| Dataset | Users | Items | Interactions | Avg Seq Len | Sparsity |
| --- | --- | --- | --- | --- | --- |
| Beauty | 22,363 | 12,101 | 198,502 | 8.88 | 99.93% |
| ML-1M | 6,040 | 3,416 | 999,611 | 165.50 | 95.16% |
| LastFM | 1,090 | 3,646 | 52,551 | 48.21 | 98.68% |

Angka LastFM cocok persis dengan CP1: 1.090 user / 52.551 interaksi / 3.646 item.

## Verifikasi & Audit

Dua tahap pemeriksaan dijalankan setelah preprocessing. Semua dataset lulus.

### `verify_output`

Tiga pemeriksaan pada file output:

- Gap tidak negatif — urutan kronologis terjaga
- Tidak ada nilai mirip timestamp absolut — threshold > 3.153.600.000 detik (100 tahun)
- Panjang sequence item dan timestamp identik per user

### `audit_alignment`

Enam pemeriksaan alignment antara `{dataset}.txt` dan `{dataset}_time.txt`:

1. Jumlah baris kedua file sama
2. User ID konsisten di kedua file tanpa ada yang hilang
3. Panjang sequence item == panjang timestamp per user
4. Interval pertama setiap user = 0
5. Semua interval ≥ 0
6. Spot-check panjang sequence vs file sumber asli HetRec — khusus LastFM, lulus 5/5

### Hasil Audit

| Dataset | Verify | Audit |
| --- | --- | --- |
| Beauty | ✅ Lulus | ✅ Lulus (spot-check dilewati, tidak ada file sumber) |
| ML-1M | ✅ Lulus | ✅ Lulus (spot-check dilewati, tidak ada file sumber) |
| LastFM | ✅ Lulus | ✅ Lulus (spot-check 5/5) |

## Catatan Penting

### Aturan Deduplikasi

Deduplikasi dilakukan pada level pasangan **(user, artist)** — satu interaksi per pasangan, ambil timestamp pertama. Deduplikasi ini dilakukan sebelum item dan timestamp dipisah untuk menjamin alignment. Aturan ini sama dengan CP1 sehingga hasil perbandingan valid.

### Recalculate Gap Setelah Filter

Setelah K-core filtering membuang event-event awal user, interval dihitung ulang dari nol menggunakan timestamp absolut yang direkonstruksi. Tanpa langkah ini, interval pertama user bisa bernilai bukan nol karena dihitung dari selisih dengan event yang sudah dibuang.

### Gap Ekstrem di LastFM

Tiga user di LastFM (user 769, 185, 546) memiliki gap ekstrem antara \~9.620–19.662 hari (\~26–54 tahun) di posisi ke-2 sequence mereka. Ini bukan bug preprocessing melainkan noise di data sumber HetRec — timestamp yang tidak konsisten di file `user_taggedartists-timestamps.dat`. Ketiga user ini tetap dipertahankan agar protokol tidak berbeda dari CP1.

### Timestamp LastFM Bukan Listening History

Timestamp di dataset LastFM HetRec 2011 adalah waktu **pemberian tag artis** (`user_taggedartists-timestamps.dat`), bukan waktu mendengarkan musik. Time interval yang dihitung mencerminkan selisih aktivitas tagging, bukan selisih waktu mendengarkan. Ini harus disebutkan sebagai limitasi dalam laporan.
